# 02 - ERM vs Group DRO Results Analysis

## Objective
Compare the final ERM and Group DRO models on the CivilComments-WILDS test set, including overall performance, worst-group accuracy, identity-label group performance, and the average-versus-worst-group trade-off.

No model training or model selection is performed in this notebook.


## 1. Imports


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
print("Imports successful")


## 2. Paths


In [ ]:
# Notebook expected in notebooks/. If run from project root, remove ../
ERM_DIR = Path("../results/erm")
DRO_DIR = Path("../results/group_dro")
COMPARISON_DIR = Path("../results/comparison")
FIGURE_DIR = Path("../figures/results")
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

required = [
    ERM_DIR / "test_metrics.json",
    ERM_DIR / "test_group_metrics.csv",
    DRO_DIR / "test_metrics.json",
    DRO_DIR / "test_group_metrics.csv",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing files:
" + "
".join(missing))
print("All required result files found.")


## 3. Load Overall Test Metrics


In [ ]:
with open(ERM_DIR / "test_metrics.json", encoding="utf-8") as f:
    erm_metrics = json.load(f)
with open(DRO_DIR / "test_metrics.json", encoding="utf-8") as f:
    dro_metrics = json.load(f)
print("ERM:", json.dumps(erm_metrics, indent=2))
print("Group DRO:", json.dumps(dro_metrics, indent=2))


## 4. Overall Comparison


In [ ]:
needed = ["accuracy", "precision", "recall", "f1", "worst_group_accuracy"]
for metric in needed:
    if metric not in erm_metrics or metric not in dro_metrics:
        raise KeyError(f"Missing metric: {metric}")

comparison_df = pd.DataFrame([
    {"Method":"ERM", "Accuracy":erm_metrics["accuracy"], "Precision":erm_metrics["precision"], "Recall":erm_metrics["recall"], "F1":erm_metrics["f1"], "Worst-group Accuracy":erm_metrics["worst_group_accuracy"]},
    {"Method":"Group DRO", "Accuracy":dro_metrics["accuracy"], "Precision":dro_metrics["precision"], "Recall":dro_metrics["recall"], "F1":dro_metrics["f1"], "Worst-group Accuracy":dro_metrics["worst_group_accuracy"]},
])
comparison_df.round(4)


## 5. Metric Changes


In [ ]:
erm_row = comparison_df.query("Method == 'ERM'").iloc[0]
dro_row = comparison_df.query("Method == 'Group DRO'").iloc[0]
metrics = ["Accuracy", "Precision", "Recall", "F1", "Worst-group Accuracy"]
changes = {m: float(dro_row[m] - erm_row[m]) for m in metrics}
changes_df = pd.DataFrame({"Metric":changes.keys(), "Change (Group DRO - ERM)":changes.values()})
display(changes_df.round(4))
for m, v in changes.items(): print(f"{m}: {v:+.4f}")


## 6. Main Figure: Average vs Worst-Group Accuracy


In [ ]:
plot_df = comparison_df[["Method","Accuracy","Worst-group Accuracy"]].melt(id_vars="Method", var_name="Metric", value_name="Score")
plt.figure(figsize=(8,5))
ax=sns.barplot(data=plot_df,x="Method",y="Score",hue="Metric")
for container in ax.containers: ax.bar_label(container,fmt="%.3f",padding=3)
plt.ylim(0,1); plt.xlabel(""); plt.ylabel("Accuracy"); plt.title("Average vs Worst-Group Test Accuracy"); plt.legend(title="")
plt.tight_layout(); plt.savefig(FIGURE_DIR/"average_vs_worst_group_accuracy.png",dpi=300,bbox_inches="tight"); plt.show()


## 7. Load and Verify Group-Level Metrics


In [ ]:
erm_groups=pd.read_csv(ERM_DIR/"test_group_metrics.csv")
dro_groups=pd.read_csv(DRO_DIR/"test_group_metrics.csv")
expected={"identity","label","label_name","n","accuracy"}
for name,df in [("ERM",erm_groups),("Group DRO",dro_groups)]:
    miss=expected-set(df.columns)
    if miss: raise ValueError(f"{name} missing columns: {sorted(miss)}")
assert set(zip(erm_groups.identity,erm_groups.label))==set(zip(dro_groups.identity,dro_groups.label))
size_check=erm_groups[["identity","label","n"]].merge(dro_groups[["identity","label","n"]],on=["identity","label"],suffixes=("_erm","_dro"),validate="one_to_one")
assert (size_check.n_erm==size_check.n_dro).all()
print("Evaluation groups and group sizes match.")


## 8. Group-Level Comparison


In [ ]:
group_comparison=erm_groups[["identity","label","label_name","n","accuracy"]].rename(columns={"accuracy":"ERM_accuracy"}).merge(
    dro_groups[["identity","label","accuracy"]].rename(columns={"accuracy":"GroupDRO_accuracy"}),
    on=["identity","label"],validate="one_to_one")
group_comparison["difference"]=group_comparison.GroupDRO_accuracy-group_comparison.ERM_accuracy
group_comparison["group_name"]=group_comparison.identity+" | "+group_comparison.label_name
erm_worst=group_comparison.sort_values("ERM_accuracy").iloc[0]
dro_worst=group_comparison.sort_values("GroupDRO_accuracy").iloc[0]
display(group_comparison.sort_values("ERM_accuracy").round(4))
print("ERM worst group:
",erm_worst[["identity","label_name","n","ERM_accuracy"]])
print("
Group DRO worst group:
",dro_worst[["identity","label_name","n","GroupDRO_accuracy"]])


## 9. Largest Improvements and Decreases


In [ ]:
print("Largest improvements")
display(group_comparison.nlargest(5,"difference")[["group_name","n","ERM_accuracy","GroupDRO_accuracy","difference"]].round(4))
print("Largest decreases")
display(group_comparison.nsmallest(5,"difference")[["group_name","n","ERM_accuracy","GroupDRO_accuracy","difference"]].round(4))


## 10. Figure: Accuracy Across Groups


In [ ]:
group_plot=group_comparison[["group_name","ERM_accuracy","GroupDRO_accuracy"]].melt(id_vars="group_name",var_name="Method",value_name="Accuracy")
group_plot.Method=group_plot.Method.replace({"ERM_accuracy":"ERM","GroupDRO_accuracy":"Group DRO"})
plt.figure(figsize=(12,8)); sns.barplot(data=group_plot,y="group_name",x="Accuracy",hue="Method")
plt.xlim(0,1); plt.xlabel("Accuracy"); plt.ylabel("Identity-label group"); plt.title("ERM vs Group DRO Accuracy Across Evaluation Groups")
plt.tight_layout(); plt.savefig(FIGURE_DIR/"group_accuracy_comparison.png",dpi=300,bbox_inches="tight"); plt.show()


## 11. Figure: Accuracy Change by Group


In [ ]:
imp=group_comparison.sort_values("difference")
colors=["green" if x>0 else "red" for x in imp.difference]
plt.figure(figsize=(10,8)); plt.barh(imp.group_name,imp.difference,color=colors); plt.axvline(0,color="black",linewidth=1)
plt.xlabel("Accuracy change (Group DRO - ERM)"); plt.title("Change in Group Accuracy After Group DRO")
plt.tight_layout(); plt.savefig(FIGURE_DIR/"group_accuracy_change.png",dpi=300,bbox_inches="tight"); plt.show()


## 12. Group DRO Validation History


In [ ]:
history_path=DRO_DIR/"training_history.csv"
if history_path.exists():
    history=pd.read_csv(history_path); display(history)
    required={"epoch","val_accuracy","val_worst_group_accuracy"}
    miss=required-set(history.columns)
    if miss: raise ValueError(f"Training history missing: {sorted(miss)}")
    plt.figure(figsize=(8,5)); plt.plot(history.epoch,history.val_accuracy,marker="o",label="Average validation accuracy"); plt.plot(history.epoch,history.val_worst_group_accuracy,marker="o",label="Worst-group validation accuracy")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("Group DRO Validation Performance"); plt.xticks(history.epoch); plt.ylim(0,1); plt.legend(); plt.tight_layout()
    plt.savefig(FIGURE_DIR/"groupdro_validation_history.png",dpi=300,bbox_inches="tight"); plt.show()
else:
    print("training_history.csv not found; skipping this figure.")


## 13. Save Tables and Print Summary


In [ ]:
comparison_df.to_csv(COMPARISON_DIR/"overall_comparison.csv",index=False)
changes_df.to_csv(COMPARISON_DIR/"metric_changes.csv",index=False)
group_comparison.to_csv(COMPARISON_DIR/"group_comparison.csv",index=False)
print("="*65); print("ERM VS GROUP DRO TEST COMPARISON"); print("="*65)
print(comparison_df.round(4).to_string(index=False))
print("
Changes (Group DRO - ERM):")
for m,v in changes.items(): print(f"  {m}: {v:+.4f}")
print(f"
ERM worst: {erm_worst.identity} | {erm_worst.label_name} | n={int(erm_worst.n)} | {erm_worst.ERM_accuracy:.4f}")
print(f"Group DRO worst: {dro_worst.identity} | {dro_worst.label_name} | n={int(dro_worst.n)} | {dro_worst.GroupDRO_accuracy:.4f}")


## 14. Interpretation

Complete this section using the generated results:

- **Average-performance effect:** [Describe test accuracy and F1 changes.]
- **Worst-group effect:** [Describe the worst-group accuracy change.]
- **Trade-off:** [State whether average performance was exchanged for worst-group performance.]
- **Most improved groups:** [List the largest measured improvements.]
- **Groups with reduced performance:** [List the measured decreases.]
- **Model-selection observation:** [Explain the Group DRO validation pattern across epochs.]

Do not describe Group DRO as universally better unless all relevant results support that conclusion.
